In [ ]:
# Imports
import pandas as pd
from pandas import DataFrame
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import tkinter as tk
import math
import numpy as np
from tkinter import filedialog
from scipy.signal import filtfilt, butter
from torque_log_analysis import resample_data,drop_rows_from_df
import os
plt.style.use('bmh')


In [ ]:


# Create the Tkinter root widget
# root = tk.Tk()
# root.withdraw()  # Hide the main window

# # Ask the user to select a file
# dyno_file_path = filedialog.askopenfilename(
#     title="Select the dyno run .txt file",
#     filetypes=[("All files", "*.*")],
#     initialdir="/",
#     message="Please select the dyno run .txt."
# )
# # Ask the user to select a file
# car_log_file_path = filedialog.askopenfilename(
#     title="Select the car log .csv file",
#     filetypes=[("All files", "*.*")],
#     initialdir="/",
#     message="Please select the car log csv."
# )
# dyno_file_path
# create list of tuple of dyno log filepath and car log filepath
log_paths = [
    ("4-26-23_power_run.txt",r"C:\Users\Matthew Samson\Documents\formula\4-26-2023_dyno_and_regen\parsed-data\parseddata0003.CSV"), # 0
    (r"emrax228HVweakAssDynoRun.txt",r"C:\Users\Matthew Samson\Documents\formula\228hv\parsed-data\parsedMDY_3-3-2024_HMS_3-58-56.CSV"), # 1
    ("asdf",r"C:\Users\Matthew Samson\Documents\formula\228hv\parsed-data\parsedMDY_3-3-2024_HMS_4-11-35.CSV") # need to find the right dyno log for this day 
]
log_index = 0


In [ ]:
# # Reformat fucked up dyno logs
# with open('dynoruns\DJDataExport_DYNORUN_015.txt', 'r') as file:
# # Read the file
#     lines = file.readlines()

# # Remove leading and trailing spaces, split columns, and exclude lines with 'N/A'
# formatted_lines = [line.strip().split() for line in lines if 'N/A' not in line]

# # Reformat the lines with tab delimiter
# formatted_text = '\n'.join([' '.join(line) for line in formatted_lines])

# # Write the reformatted text back to the file
# with open('your_file_reformatted.txt', 'w') as file:
#     file.write(formatted_text)

# This code is to reformat fucked up dyno logs if necessary



In [ ]:
# Calculate some constants 
MOTOR_KT = 0.94
gear_tooth_count_motor = 10  # KS6e front sprocket
gear_tooth_count_wheel = 30  # KS6e rear sprocket

dyno_tire_diameter = 20.5 # ish inches 
# tire_dims = [18,19,20]
tire_dims = []
normal_tire_diameter = 18 # ish inches
tire_diameter = normal_tire_diameter # set to the right tire size when plotting

gear_ratio = gear_tooth_count_wheel / gear_tooth_count_motor

# Calculate wheel circumference
wheel_circumference = tire_diameter * 3.141592653589793  # Pi
wheel_circums_varied = []
for dim in tire_dims:
    wheel_circums_varied.append(dim*3.141592653589793)
wheel_circums_varied


In [ ]:
# open dyno run file
dynofile = (os.path.basename(log_paths[log_index][0]))
dyno_run_df = pd.read_csv(log_paths[log_index][0],delimiter=" ")
# drop shit rows
dynorun = dyno_run_df[dyno_run_df["s"]>=0]
# dynorun = dyno_run_df[dyno_run_df["hp"]>=0]
dynorun=dynorun[['s','hp','mph']]
del(dyno_run_df)
dynorun["dyno_kw"] = dynorun['hp']/1.341 # add KW column
dynorun["wheel_rpm"] = dynorun["mph"] / ((wheel_circumference * 60)/63360 )
list_of_varied_dims_rpms=[]
for index,circum in enumerate(tire_dims):
    name = "wheel_rpm_"+str(circum)
    dynorun[name] = dynorun["mph"] / ((circum * 3.141592653589793 * 60)/63360 )
    list_of_varied_dims_rpms.append(name)
    
print(list_of_varied_dims_rpms)
dynorun["wheel_torque"] = (9548.8 * dynorun["dyno_kw"])/dynorun["wheel_rpm"]

list_of_varied_wtrq = []
for index,rpm in enumerate(list_of_varied_dims_rpms):
    wtrqname = "wheel_torque_"+str(tire_dims[index])
    dynorun[wtrqname]=(9548.8 * dynorun["dyno_kw"])/dynorun[rpm]
    list_of_varied_wtrq.append(wtrqname)
    
dynorun["motor_rpm"]=dynorun["wheel_rpm"]*gear_ratio

for index,rpm in enumerate(list_of_varied_dims_rpms):
    colname = "motor_rpm_" +str(tire_dims[index])
    dynorun[colname]=dynorun[rpm]*gear_ratio
    
dynorun["motor_torque"] = dynorun["wheel_torque"] / gear_ratio

for index,trq in enumerate(list_of_varied_wtrq):
    colname = "motor_torque_" +str(tire_dims[index])
    dynorun[colname]=dynorun[trq]/gear_ratio
    
print("dynorun MAXES:")
dynorun.max()

In [ ]:
# plot dyno stuff
plt.plot(dynorun['mph'],dynorun['dyno_kw'])
plt.show()

In [ ]:

# load electrical log
carfile = (os.path.basename(log_paths[log_index][1]))

vehicle_log = pd.read_csv(log_paths[log_index][1])

# remove df cols we dont care about
car_log_cols_to_keep = [
    "Time", "D1_Commanded_Torque", "D2_Torque_Feedback", "D2_Motor_Speed",
    "Pack_Current", "Pack_Inst_Voltage", "D4_Iq", "D3_Id",
    "D2_Output_Voltage", "D3_VAB_Vd_Voltage", "D4_VBC_Vq_Voltage",
    "D1_DC_Bus_Voltage","D4_DC_Bus_Current",
    "D1_Motor_Angle_Electrical"]

vehicle_log = vehicle_log[car_log_cols_to_keep]

# cancer time
vehicle_log = resample_data(vehicle_log, "Time", plots_to_compare=False)
vehicle_log = drop_rows_from_df(vehicle_log,"D1_Commanded_Torque",1,400)


vehicle_log = drop_rows_from_df(vehicle_log, "D2_Motor_Speed", 10, 7000)

# get a power column to compare with dyno
vehicle_log["kw"] = vehicle_log["Pack_Current"] * \
    vehicle_log["Pack_Inst_Voltage"] / 1000

# Define the filter
order = 2
fs = 1 / (10 / 1000)  # Sampling frequency
nyquist = fs / 2
cutoff = min(0.5 * nyquist, 5)  # Desired cutoff frequency of the filter (Hz)
print(str(cutoff)+"Hz")
b, a = butter(order, cutoff / (fs / 2), btype='low')

# Apply filtfilt to the 'Value' column
vehicle_log['smooth_trq'] = filtfilt(b, a, vehicle_log['D2_Torque_Feedback'])
vehicle_log["acc_kw"] = filtfilt(b, a, vehicle_log['kw'])
vehicle_log["smooth_rpm"] = filtfilt(b, a, vehicle_log['D2_Motor_Speed'])
vehicle_log["smooth_iq"] = filtfilt(b, a, vehicle_log['D4_Iq'])
vehicle_log["smooth_id"] = filtfilt(b, a, vehicle_log['D3_Id'])
vehicle_log["smooth_outputvoltage"] = filtfilt(
    b, a, vehicle_log['D2_Output_Voltage'])
vehicle_log["smooth_vq"] = filtfilt(b, a, vehicle_log['D4_VBC_Vq_Voltage'])
vehicle_log['smooth_vd'] = filtfilt(b, a, vehicle_log['D3_VAB_Vd_Voltage'])
vehicle_log['smooth_dcbusvoltage'] = filtfilt(b, a, vehicle_log['D1_DC_Bus_Voltage'])
vehicle_log['smooth_dcbuscurrent'] = filtfilt(b, a, vehicle_log['D4_DC_Bus_Current'])
vehicle_log['calc_motor_torque'] = (vehicle_log['smooth_iq'] / (math.sqrt(2))) * MOTOR_KT
vehicle_log['inverter_pwr_kw'] = vehicle_log['smooth_dcbusvoltage'] * vehicle_log['smooth_dcbuscurrent'] / 1000
# calculate total motor current
vehicle_log["motor_current"] = (
    vehicle_log["smooth_id"] ** 2 + vehicle_log["smooth_iq"] ** 2).pow(1/2)
vehicle_log["motor_power"] = vehicle_log['motor_current'] * vehicle_log['smooth_outputvoltage'] / 1000
# power factor
power_factor = 1.0
vehicle_log["chatgpt_power"] = (vehicle_log['smooth_vd'] * vehicle_log['smooth_id'] +
                                vehicle_log['smooth_vq'] * vehicle_log['smooth_iq'])/power_factor/1000
vehicle_log['torque_feedback_pwr_kw'] = vehicle_log['smooth_trq'] * vehicle_log['smooth_rpm'] / 9548.8
vehicle_log['motor_power_kt_calc'] = vehicle_log["calc_motor_torque"] * vehicle_log['smooth_rpm'] / 9548.8
vehicle_log.max()

In [ ]:
# Attempt to sync dyno and car log
dynorun["s"]=pd.to_datetime(dynorun["s"],unit="s")

# this offset has to be manually adjusted lol :(
runoffset = pd.tseries.offsets.DateOffset(milliseconds=0)
# sync them by finding the time where each max occured and aligning those
dynosyncindex = "dyno_kw"
carsyncindex = "acc_kw"
dynomaxindex = dynorun[dynosyncindex].idxmax()

dynomaxtimestamp = (dynorun["s"][dynomaxindex])
carmaxtimestamp = (vehicle_log[carsyncindex].idxmax())
timeoffset = carmaxtimestamp - (dynomaxtimestamp - runoffset)

dynorun["s"] = dynorun["s"]+timeoffset

dynorun

dynomaxindex = dynorun[dynosyncindex].idxmax()

print(dynorun["s"][dynomaxindex])
print(vehicle_log[carsyncindex].idxmax())

In [ ]:
# get the Xlims of start and end of dyno run (approx)
dynorunduration = dynorun["s"].max() - dynorun["s"].min()
xlimstart = carmaxtimestamp - dynorunduration/2
xlimend = carmaxtimestamp + dynorunduration*2

xlims = (xlimstart,xlimend)
xlims

# attempt to drop data outside of this range
vehicle_log  = vehicle_log[(vehicle_log.index > xlims[0])]
vehicle_log = vehicle_log[(vehicle_log.index < xlims[1])]
vehicle_log.max()

In [ ]:
# plot Inverter and BMS current for sanity
plt.figure(figsize=(16,7))
plt.scatter(vehicle_log.index,vehicle_log['smooth_dcbuscurrent'],label='smooth_dcbuscurrent',marker='.')
plt.scatter(vehicle_log.index,vehicle_log['Pack_Current'],label='Pack_Current',marker='.')
plt.legend()
plt.show()

In [ ]:
# plot Inverter vs BMS voltage for sanity
plt.figure(figsize=(16,7))
plt.scatter(vehicle_log.index,vehicle_log['smooth_dcbusvoltage'],marker='.',label='smooth_dcbusvoltage')
plt.scatter(vehicle_log.index,vehicle_log['Pack_Inst_Voltage'],marker='.',label='bmsvoltage')
plt.legend()
plt.ylim(0,310)
plt.show()

In [ ]:
plt.plot(vehicle_log.index,vehicle_log['calc_motor_torque'],label='Iq(rms) * Kt ('+str(MOTOR_KT)+')')
plt.plot(vehicle_log.index,vehicle_log['smooth_trq'],label='motor torque feedback')
plt.legend()
plt.show()

In [ ]:
# Attempt to plot motor current, power, etc. etc.
fig,ax = plt.subplots(figsize=(20,10))

ax.scatter(vehicle_log.index,vehicle_log['chatgpt_power'],label="sqrt(Vq*Iq + Vd*Id) /"+str(power_factor)+"(power factor)",marker='.')
ax.scatter(vehicle_log.index,(vehicle_log["motor_power"]),marker='.',label="motor current * motor voltage")
# ax.scatter(vehicle_log.index,vehicle_log['evwest_guide_kw']/1000,marker='.',label='(vq * iq + vd * id / power_factor)')
ax.scatter(vehicle_log.index,vehicle_log['acc_kw'],marker='.',label="Accumulator Electrical Power (Current * Voltage)")
ax.scatter(vehicle_log.index,vehicle_log['inverter_pwr_kw'],marker='.',label='Inverter Electrical Power (DC Bus Current * DC Bus Voltage)')
ax.scatter(vehicle_log.index,vehicle_log['torque_feedback_pwr_kw'],marker='.',label='Inverter Torque Feedback * RPM (Mech Power)')
ax.scatter(vehicle_log.index,vehicle_log['motor_power_kt_calc'],marker='.',label='Calculated Torque * RPM (Mech Power)')

# plt.scatter(vehicle_log.index,vehicle_log['motor_current']/10)
filenames=f"\nDyno log: {dynofile} Car log: {carfile}"

def annot_cols(df:DataFrame,cols:list,ax,unit:str,timeindex=None):
    for col in cols:
        xmax = (df[col].idxmax())
        if timeindex:
            xmax = df[timeindex][xmax]
        ymax = df[col].max()
        ax.annotate(
            f"{col} max: {ymax:.2f}{unit}",
            xy=(xmax, ymax), xycoords='data',
            xytext=(-300, 0), textcoords='offset points',
            arrowprops=dict(arrowstyle='->',linewidth=1.2,color='black'))
        
cols = [
    'chatgpt_power','motor_power','acc_kw','inverter_pwr_kw','torque_feedback_pwr_kw','motor_power_kt_calc'
]
annot_cols(vehicle_log,cols,ax,"kW")
plt.tight_layout()
plt.xlabel("Time")
plt.ylabel("Kilowatts")
plt.legend()
plt.title("Lots of different power levels"+filenames)
plt.show()

In [ ]:
# plot torques and rpms to compare

fig,ax = plt.subplots(figsize=(16,7))
ax.plot(dynorun["motor_rpm"],dynorun["motor_torque"],label="Dyno Measured Torque",color='y')

for dim in tire_dims:
    ax.plot(dynorun["motor_rpm_"+str(dim)],dynorun["motor_torque_"+str(dim)],label="motor_torque_"+str(dim))
        
# ax.scatter(vehicle_log["D2_Motor_Speed"],vehicle_log["D2_Torque_Feedback"],marker='.',color='indigo',label="PM100 Torque Feedback")
ax.scatter(vehicle_log["D2_Motor_Speed"],vehicle_log["smooth_trq"],marker='.',label="Smoothed Torque feedback")

plt.xlabel("Motor RPM")
plt.ylabel("Torque (Newton-meters)")
plt.legend()
plt.title("KS6E Torque vs. RPM"+filenames)
plt.show()

In [ ]:
fig,ax = plt.subplots(figsize=(16,9))
ax.plot(dynorun["s"],dynorun['motor_torque'],label="DynoTorque",c="r")
for dim in tire_dims:
    ax.plot(dynorun["s"],dynorun["motor_torque_"+str(dim)],label="motor_torque_"+str(dim))
  
ax.plot(vehicle_log.index,vehicle_log['smooth_trq'],label="TorqueFeedback",c="b")
plt.title("Dyno Torque and Inverter Torque Feedback vs. Time"+filenames)
plt.ylabel("Torque(Nm)")
plt.xlabel("Time")

annot_cols(dynorun,["motor_torque"],ax,"Nm",'s')
annot_cols(vehicle_log,["smooth_trq"],ax,"Nm")
plt.tight_layout()
plt.legend(loc='upper right')
plt.show()

In [ ]:
fig = plt.figure()
plt.plot(vehicle_log.index,vehicle_log["smooth_rpm"],label="Motor Resolver Feedback RPM Reading")
plt.plot(dynorun["s"],dynorun["motor_rpm"],label="Motor RPM Calculated from Dyno Speed ("+str(tire_diameter)+"inch)")
for dim in tire_dims:
    plt.plot(dynorun["s"],dynorun["motor_rpm_"+str(dim)],label="Calculated with " + str(dim)+"inch")

plt.title("Dyno RPM and Inverter RPM Feedback vs. Time"+filenames)
plt.ylabel("RPM")
plt.xlabel("Time")

plt.legend(loc='upper left')
plt.show()


In [ ]:
# attempt to combine dyno and car dataframes
df = dynorun.set_index('s')
combined_df = vehicle_log.join(df)
combined_df = combined_df.astype(float)
combined_df = resample_data(combined_df,plots_to_compare=False)
combined_df.max()

In [ ]:
# calculate power error. rpm
combined_df["acc_power_delta"] = (combined_df["dyno_kw"] - combined_df["acc_kw"]).abs()
combined_df["inv_mech_power_delta"] = (combined_df["dyno_kw"] - combined_df["torque_feedback_pwr_kw"]).abs()

# combined_df['power_error'] = (combined_df["acc_power_delta"] / combined_df['acc_kw'])

# combined_df["power_error"] = ((combined_df["acc_kw"] - combined_df["dyno_kw"]) ** 2).mean() ** .5
combined_df



In [ ]:
fig = plt.figure(figsize=(16, 9))
plt.scatter(combined_df.index,combined_df['acc_power_delta'],label='Difference between acc kw and dyno kw',marker='.')
# plt.scatter(combined_df.index,combined_df['acc_power_delta']*1.341,label='Delta Horsies',marker='.')

plt.scatter(combined_df.index,combined_df['inv_mech_power_delta'],label='Difference between dyno kw and Inverter mech kw',marker='.')
# plt.scatter(combined_df.index,combined_df['inv_mech_power_delta']*1.341,label='Inv Delta Horsies',marker='.')

# plt.scatter(combined_df.index,combined_df['power_error'],label='Smoothed Acc Power',marker='.')

plt.ylabel("Kilowatts/Horsepower")
plt.xlabel("Time")
plt.xlim(xlims)
# plt.ylim((-1,40))

plt.title("Power delta between Acc output and Dyno measured"+filenames)
plt.legend(loc="upper left")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(16,9))

ax.scatter(combined_df.index,combined_df['acc_kw'],label='Accumulator Electrical Power Output (Current * Voltage)',marker='.')
ax.scatter(combined_df.index,combined_df['torque_feedback_pwr_kw'],marker='.',label='Inverter mech power (Torque Feedback * RPM)')
# ax.scatter(combined_df.index,combined_df['motor_power_kt_calc'],marker='.',label='Calc mech power (calc Torque  * RPM)')
ax.scatter(combined_df.index,combined_df['inverter_pwr_kw'],marker='.',label='Inverter Electrical Power (DC Bus Current * Voltage)')
ax.scatter(combined_df.index,combined_df['dyno_kw'],label='Dyno Measured Power',marker='.')
ax.set_ylabel("Kilowatts")
ax.set_xlabel("Time")

cols = [
    'acc_kw','torque_feedback_pwr_kw','inverter_pwr_kw','dyno_kw'
]
annot_cols(combined_df,cols,ax,"kW")

plt.title("Acc & Dyno Power vs Time"+filenames)
plt.legend(loc="upper right")
plt.show()

In [ ]:
def calculate_efficiency(input_value, output_value):
    # print(f"In: {input_value:.2f} Out: {output_value:.2f}")
    efficiency = (output_value / input_value)
    # print(f"Efficiency {efficiency*100:.2f}%")
    return efficiency
acc_pwr_max = combined_df['acc_kw'].max()
motor_pwr_max = combined_df['torque_feedback_pwr_kw'].max()
dyno_pwr_max = combined_df['dyno_kw'].max()
stage1eff = calculate_efficiency(acc_pwr_max,motor_pwr_max)
stage2eff = calculate_efficiency(motor_pwr_max,dyno_pwr_max)
print("Assuming all peak power were hit at the same point")
print(f"Acc power max: \n\t{acc_pwr_max:.2f}kW\nMotor power max: \n\t{motor_pwr_max:.2f}kW\nDyno power max: \n\t{dyno_pwr_max:.2f}kW")
print(f"Acc to motor efficiency: \n\t{stage1eff*100:.2f}%")
print(f"Motor to dyno efficiency: \n\t{stage2eff*100:.2f}%")
print(f"Total efficiency \n\t{stage1eff*stage2eff*100}%")

In [ ]:
# Estimating a dyno error factor
expected_motor_to_wheels_efficiency = 0.95 # What we expect the drivetrain loss to be
# Do we factor in powerloss from tires? Is that a thing?
expected_dyno_pwr_max = expected_motor_to_wheels_efficiency * motor_pwr_max
dyno_error_kw = expected_dyno_pwr_max-dyno_pwr_max
dyno_error_kw
print(f"Dyno highest power reading: \n\t{dyno_pwr_max:.2f}kW")
print(f"Expected power reading with {expected_motor_to_wheels_efficiency*100}% drivetrain efficiency: \n\t{expected_dyno_pwr_max:.2f}kW")
print(f"Error of dyno: \n\t{dyno_error_kw:.2f}kW")
print(f"In horsepower: \n\t{dyno_error_kw*1.341:.2f}HP")